MANC notebook- create csv annotation fils accompanying neuroglancer links from MANC version v1.2. Jelly Soffers 20260309

In [1]:
# set up environment
import sys
import json
import re
import requests
import pandas as pd
from pathlib import Path
import json

from caveclient import CAVEclient

print("Python executable:", sys.executable)
print("Imports OK")

c:\Users\JHS\Documents\Python\cave_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Python executable: c:\Users\JHS\Documents\Python\cave_env\Scripts\python.exe
Imports OK


In [2]:
#set up a output directory for tables etc.


PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output:", OUTPUT_DIR)


STATE_PATH = OUTPUT_DIR / "manc_v1p2_with_manual_IR_layers.json"

with open(STATE_PATH, "r", encoding="utf-8") as f:
    state = json.load(f)

#alternative to open from a clio link:
##import json
##import urllib.parse

##clio_url = "https://clio-ng.janelia.org/#!<your-encoded-json-here>"
##encoded = clio_url.split("#!")[-1]
##state = json.loads(urllib.parse.unquote(encoded))


Project root: c:\Users\JHS\Documents\Python\project_folder_4B
Output: c:\Users\JHS\Documents\Python\project_folder_4B\output


In [ ]:
#build a simple descriptive CSV to describe the state view
import json
import csv
from pathlib import Path
import re

OUTPUT_DIR = Path(OUTPUT_DIR)
STATE_PATH = OUTPUT_DIR / "manc_v1p2_with_manual_IR_layers.json"

# Load state
with open(STATE_PATH, "r", encoding="utf-8") as f:
    state = json.load(f)

state_title = state.get("title", "untitled_state")

# Clean title for filename (remove spaces/special chars)
safe_title = re.sub(r"[^a-zA-Z0-9_-]+", "_", state_title)

OUT_CSV = OUTPUT_DIR / f"{safe_title}_layer_annotations.csv"

rows = []

for L in state.get("layers", []):
    layer_type = L.get("type", "")
    name = L.get("name", "")
    visible = L.get("visible", "")

    segments = L.get("segments", [])
    segments_csv = ",".join(str(s) for s in segments) if segments else ""

    seg_colors = L.get("segmentColors") or {}
    unique_colors = set(seg_colors.values())
    layer_color = list(unique_colors)[0] if len(unique_colors) == 1 else ""

    rows.append({
        "state_title": state_title,
        "layer_name": name,
        "layer_type": layer_type,
        "visible": visible,
        "layer_color": layer_color,
        "num_segments": len(segments),
        "segments_csv": segments_csv,
        "note": ""
    })

fieldnames = [
    "state_title",
    "layer_name",
    "layer_type",
    "visible",
    "layer_color",
    "num_segments",
    "segments_csv",
    "note",
]

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("Wrote:", OUT_CSV)

Wrote: c:\Users\JHS\Documents\Python\project_folder_4B\output\4B_analysis_manual_IR_layers_layer_annotations.csv


In [6]:
# Build unified DataFrame for all layers (ready for annotation)
import json
import re
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(OUTPUT_DIR)
STATE_PATH = OUTPUT_DIR / "manc_v1p2_with_manual_IR_layers.json"

# --- load state ---
with open(STATE_PATH, "r", encoding="utf-8") as f:
    state = json.load(f)

state_title = state.get("title", "untitled_state")
safe_title = re.sub(r"[^A-Za-z0-9_-]+", "_", state_title)

rows = []

for L in state.get("layers", []):
    layer_name = L.get("name", "<unnamed>")
    layer_type = L.get("type", "")
    visible = L.get("visible", "")
    tab = L.get("tab", "")
    segment_query = L.get("segmentQuery", None)

    body_ids = L.get("segments", []) or []
    segments_csv = ",".join(str(int(s)) for s in body_ids) if body_ids else ""
    bodyId_preview = ",".join(str(int(s)) for s in body_ids[:20]) if body_ids else ""

    seg_colors = L.get("segmentColors") or {}
    unique_colors = set(
        seg_colors.get(str(int(s))) for s in body_ids if str(int(s)) in seg_colors
    )
    unique_colors = {c for c in unique_colors if c}
    layer_color = list(unique_colors)[0] if len(unique_colors) == 1 else ""

    rows.append({
        "state_title": state_title,
        "layer_name": layer_name,
        "layer_type": layer_type,
        "visible": visible,
        "tab": tab,
        "segmentQuery": segment_query,
        "layer_color": layer_color,
        "num_segments": len(body_ids),
        "segments_csv": segments_csv,
        "bodyId_preview": bodyId_preview,
        "notes": ""   # <-- ready for manual annotation
    })

summary_df = (
    pd.DataFrame(rows)
    .sort_values(["layer_type", "num_segments"], ascending=[True, False])
    .reset_index(drop=True)
)

summary_df.head(3)

#export as .csv
##OUT_CSV = OUTPUT_DIR / f"{safe_title}_layer_annotations.csv"
##summary_df.to_csv(OUT_CSV, index=False)
##print("Wrote:", OUT_CSV)

,state_title,layer_name,layer_type,visible,tab,segmentQuery,layer_color,num_segments,segments_csv,bodyId_preview,notes
0,4B analysis + manual IR layers,presyn,annotation,False,rendering,None,,0,,,
1,4B analysis + manual IR layers,postsyn,annotation,False,rendering,None,,0,,,
2,4B analysis + manual IR layers,em,image,,source,None,,0,,,


In [8]:
#Step 3: Preparation for making my notes dictionary that will popualte the notes column of summary df. Dun this cell to print the segmentation layer names. Manually copy this output and add your notes betweenthe '' in the next cell
# Print / build the keys you need to annotate
NOTE_KEY_COL = "layer_name"   # <-  summary_df uses this

NOTES = {name: "" for name in summary_df[NOTE_KEY_COL].dropna().unique()}
NOTES

{'presyn': '',
 'postsyn': '',
 'em': '',
 'manc-seg-v1.2-4B-all': '',
 'manc-seg-v1.21- secondary n=73': '',
 'nerves': '',
 'neuropils': '',
 'manc-seg-v1.22-4B-early secondary n= 20': '',
 'manc-seg-v1.2 4B manual IR-1': '',
 'manc-seg-v1.2-4B-subbclass BI n=4': '',
 'manc-seg-v1.2 4B manual BR-2 n=10': '',
 'manc-seg-v1.2 4B manual IR-2': '',
 'court et al. tracts': '',
 'manc-seg-v1.2 4B manual  IR-nomorphology assigned-early secondary': '',
 'manc-seg-v1.2 4B manual IR nomorphology assigned secondary': '',
 'manc-seg-v1.21-4B-subclassBR n=26': '',
 'manc-seg-v1.2 4b-manual BR-4 n=6': '',
 'manc-seg-v1.2 4B manual BR-1 n=6': '',
 'manc-seg-v1.2 4B manual IR-4': '',
 'manc-seg-v1.22-4B subclass IA=4': '',
 'manc-seg-v1.2 4B- primary  n=4': '',
 'manc-seg-v1.2 4B manual BR-3 n=4': '',
 'manc-seg-v1.2 4B manual IR-independent leg n=11': '',
 'manc-seg-v1.2 4B manual  IR- proprioceptice tactile n=3': '',
 'manc-seg-v1.2 4B manual IR-3': '',
 'manc:v1.2.1-TTMn': '',
 'all-tissue': '',


In [10]:
# Manually complete the notes in this cell in case required between the last before the  coma ''
NOTES = {
  'presyn': '',
  'postsyn': '',
  'em': '',
  'manc-seg-v1.2-4B-all': 'query: All 4B neurons selected by #hemilineage, soma side and thoracic level',
  'manc-seg-v1.21- secondary n=73': 'Secondary birthtime 4B neurons (query-derived)',
  'nerves': 'Anatomical context layer showing nerve ROIs',
  'neuropils': 'Neuropil segmentation for spatial orientation',
  'manc-seg-v1.22-4B-early secondary n= 20': 'Early secondary 4B neurons (query-derived)',
  'manc-seg-v1.2 4B manual IR-1': 'Manual morphology review group IR-no morphology assigned-group-1, posterior soma',
  'manc-seg-v1.2-4B-subbclass BI n=4': 'Subclass BI 4B neurons (query-derived)',
  'manc-seg-v1.2 4B manual BR-2 n=10': 'Manual morphology review group BR-2',
  'manc-seg-v1.2 4B manual IR-2': 'Manual morphology review group IR-no morphology assigned-group-2, anterior soma',
  'court et al. tracts': 'Published tract annotations (context only)',
  'manc-seg-v1.2 4B manual  IR-nomorphology assigned-early secondary': 'IR group, no morphology assigned, early secondary, ',
  'manc-seg-v1.2 4B manual IR nomorphology assigned secondary': 'IR group, no morphology assigned, secondary, (filter query by birth order)',
  'manc-seg-v1.21-4B-subclassBR n=26': 'All BR subclass neurons, (query-derived)',
  'manc-seg-v1.2 4b-manual BR-4 n=6': 'Manual morphology group BR-4 (manual morphology review within query-derived BR group)',
  'manc-seg-v1.2 4B manual BR-1 n=6': 'Manual morphology group BR-1 (manual morphology review within query-derived BR group)',
  'manc-seg-v1.2 4B manual IR-4': 'Manual morphology review group IR-no morphology assigned-group-4, strong anterio-posterior primary neurite',
  'manc-seg-v1.22-4B subclass IA=4': 'Subclass IA neurons (query-derived)',
  'manc-seg-v1.2 4B- primary  n=4': 'Primary birthtime 4B neurons, (filter query birth order)',
  'manc-seg-v1.2 4B manual BR-3 n=4': 'Manual morphology group BR-3,(manual morphology review within query-derived BR group)',
  'manc-seg-v1.2 4B manual IR-independent leg n=11': 'IR neurons independent of leg motor pools, (query-derived)',
  'manc-seg-v1.2 4B manual  IR- proprioceptice tactile n=3': 'IR proprioceptive/tactile subtype, (manual morphology review filter by query subtype',
  'manc-seg-v1.2 4B manual IR-3': 'Manual morphology review group IR-no morphology assigned-group-1, lateral soma',
  'manc:v1.2.1-TTMn': 'TT motor neuron segmentation reference',
  'all-tissue': 'Whole tissue segmentation (context only)',
  'all-synaptic': 'All synaptic neuropil regions (context only)',
  'manc-seg-v1.2-4B-sublass IR n=60': 'All IR subclass neurons, (query-derived)',
  'manc-seg-v1.21-4B subclass BA n=1': 'BA subclass neuron, (query-derived)',
  'voxel-classes': 'Voxel classification layer (technical)',
  'initial-supervoxels': 'Initial segmentation supervoxels (technical)'
}

In [12]:
#complete the table: Fill out the summary notes column with the dictionary, but save it as manc_layer_notescsv.
# Map notes into dataframe
summary_df["notes"] = summary_df["layer_name"].map(NOTES).fillna("")

# Preview
summary_df[["layer_name", "notes"]]

# Build filename with state title
safe_title = re.sub(r"[^A-Za-z0-9_-]+", "_", state_title)
OUT_CSV = OUTPUT_DIR / f"{safe_title}_layer_notes.csv"

# Save
summary_df.to_csv(OUT_CSV, index=False)

print("Saved to:", OUT_CSV)

Saved to: c:\Users\JHS\Documents\Python\project_folder_4B\output\4B_analysis_manual_IR_layers_layer_notes.csv


In [ ]:
# # Build New State From Base + New Layers
# import json
# import urllib.parse
# from copy import deepcopy
# from pathlib import Path

# OUTPUT_DIR = Path(OUTPUT_DIR)  # keep using your existing OUTPUT_DIR variable

# # 1. Load base state fresh (do not trust in-memory state)
# STATE_IN = OUTPUT_DIR / "manc_v1p2_view_state.json"  # starting state
# with open(STATE_IN, "r", encoding="utf-8") as f:
#     base_state = json.load(f)

# state = deepcopy(base_state)

# # 2. Define your new segmentation layers
# TEST_LAYERS = {
#     "manc-seg-v1.2 4B manual IR-1": [102536, 102138, 19833, 21189, 21322, 24181, 18253, 18629, 27760, 164146, 26941, 38071, 22868, 23716, 18156],
#     "manc-seg-v1.2 4B manual IR-2": [22163, 100525, 153878, 21499, 18641, 29988, 165318, 22997, 18990],
#     "manc-seg-v1.2 4B manual IR-3": [17935, 17567],
#     "manc-seg-v1.2 4B manual IR-4": [20077, 21567, 101453, 17572, 18785],
# }

# # --- color palette (one color per layer, will cycle if more layers than colors) ---
# # Sky blue, Orange, Vermillion, Pale violet
# COLORS = ["#87CEEB", "#FFA500", "#E34234", "#CC99FF"]

# # 3. Find the correct MANC segmentation source
# seg_source = None
# for L in state["layers"]:
#     if L.get("type") == "segmentation":
#         src = L.get("source")
#         url = src.get("url") if isinstance(src, dict) else src
#         if "manc-seg-v1p2/manc-seg-v1.2" in str(url):
#             seg_source = src
#             break

# if seg_source is None:
#     raise ValueError("Could not find MANC segmentation source.")

# # 4. Append new layers, assigning a fixed color per layer
# for i, (name, body_ids) in enumerate(TEST_LAYERS.items()):
#     layer_color = COLORS[i % len(COLORS)]
#     state["layers"].append({
#         "type": "segmentation",
#         "source": seg_source,
#         "tab": "segments",
#         "name": name,
#         "segments": [str(int(b)) for b in body_ids],
#         "segmentColors": {str(int(b)): layer_color for b in body_ids},
#         "visible": True,
#     })

# # 5. Update title BEFORE saving and encoding
# state["title"] = "4B analysis + manual IR layers"

# # 6. Save new JSON (no spaces in filename)
# STATE_OUT = OUTPUT_DIR / "manc_v1p2_with_manual_IR_layers.json"
# with open(STATE_OUT, "w", encoding="utf-8") as f:
#     json.dump(state, f, indent=2)

# # 7. Generate URL from THIS state
# encoded = urllib.parse.quote(json.dumps(state, separators=(",", ":")))
# NEW_URL = "https://clio-ng.janelia.org/#!" + encoded

# print("Saved new state:", STATE_OUT)
# print("\nNew URL:\n")
# print(NEW_URL)

Saved new state: c:\Users\JHS\Documents\Python\project_folder_4B\output\manc_v1p2_with_manual_IR_layers.json

New URL:

https://clio-ng.janelia.org/#!%7B%22title%22%3A%224B%20analysis%20%2B%20manual%20IR%20layers%22%2C%22dimensions%22%3A%7B%22x%22%3A%5B8e-09%2C%22m%22%5D%2C%22y%22%3A%5B8e-09%2C%22m%22%5D%2C%22z%22%3A%5B8e-09%2C%22m%22%5D%7D%2C%22position%22%3A%5B15353.5%2C35844.5%2C53248.5%5D%2C%22crossSectionOrientation%22%3A%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22%3A1%2C%22projectionOrientation%22%3A%5B-0.29617950320243835%2C0.698911726474762%2C-0.6356714367866516%2C0.14043490588665009%5D%2C%22projectionScale%22%3A26672.2540375702%2C%22layers%22%3A%5B%7B%22type%22%3A%22image%22%2C%22source%22%3A%7B%22url%22%3A%22precomputed%3A//gs%3A//flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22%3A%7B%22default%22%3Atrue%7D%2C%22enableDefaultSubsources%22%3Afalse%7D%2C%22tab%22%3A%22source%22%2C%22name%22%3A%22em